# Bloco 4 — Estacionariedade, cointegração e quebras estruturais

## Objetivo

Determinar a ordem de integração de cada série e investigar cointegração entre as 
variáveis para decidir a forma correta de especificação do modelo (VAR em níveis, 
VAR em diferenças, ou VECM).

## 1. Testes de estacionariedade — séries em nível

### Metodologia

Aplicamos dois testes complementares a cada série:

- **ADF (Augmented Dickey-Fuller)**: H₀ = série tem raiz unitária (não-estacionária)
- **KPSS (Kwiatkowski-Phillips-Schmidt-Shin)**: H₀ = série é estacionária

Os dois testes têm hipóteses nulas opostas, e ler ambos em conjunto reduz 
ambiguidade. Interpretação:

| ADF | KPSS | Conclusão |
|-----|------|-----------|
| Rejeita H₀ | Não rejeita H₀ | **Caso A**: série é I(0) (estacionária) |
| Não rejeita H₀ | Rejeita H₀ | **Caso B**: série é I(1) ou superior |
| Rejeita H₀ | Rejeita H₀ | **Caso C**: conflito (testar quebra estrutural) |
| Não rejeita H₀ | Não rejeita H₀ | **Caso D**: inconclusivo |

### Especificação determinística

A especificação da equação de teste depende do comportamento visual da série. 
Séries com tendência clara usam constante + tendência ("ct"); séries com média 
não-zero mas sem tendência usam apenas constante ("c").

| Série | Especificação | Justificativa |
|-------|--------------|---------------|
| `ln_ibcbr` | ct | Tendência crescente clara |
| `ln_cambio` | ct | Tendência crescente apesar de quebra em 2014-15 |
| `ln_commodities` | ct | Tendência crescente clara |
| `ln_m1` | ct | Tendência crescente forte |
| `ln_prod_industrial` | c | Estável global, mas com possível quebra em 2014-15 |
| `ln_credito_total` | ct | Tendência crescente forte |
| `ipca` | c | Variação % com média ≠ 0, sem tendência |
| `selic` | c | Taxa com nível variável mas sem tendência clara |
| `exp_ipca_12m` | c | Taxa com média ≠ 0, sem tendência |

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from statsmodels.tsa.stattools import adfuller, kpss
import warnings

# Silencia warnings de KPSS quando p-valor estoura limites tabulados
warnings.filterwarnings('ignore', category=Warning, module='statsmodels')

DATA_PROCESSED = Path('../data/processed')

# Carrega o dataset tratado do Bloco 3
df = pd.read_csv(
    DATA_PROCESSED / 'series_tratadas.csv',
    index_col=0,
    parse_dates=True
)

# Separa séries macroeconômicas das dummies
series_macro = [c for c in df.columns if not c.startswith('d_')]
print(f"Séries a testar ({len(series_macro)}):")
for s in series_macro:
    print(f"  - {s}")

Séries a testar (9):
  - ln_ibcbr
  - ln_cambio
  - ln_commodities
  - ln_m1
  - ln_prod_industrial
  - ln_credito_total
  - ipca
  - selic
  - exp_ipca_12m


In [2]:
# Dicionário com especificação determinística de cada série
# 'c' = constante apenas | 'ct' = constante + tendência
especificacao = {
    'ln_ibcbr':           'ct',
    'ln_cambio':          'ct',
    'ln_commodities':     'ct',
    'ln_m1':              'ct',
    'ln_prod_industrial': 'c',
    'ln_credito_total':   'ct',
    'ipca':               'c',
    'selic':              'c',
    'exp_ipca_12m':       'c',
}

# Verifica que todas as séries têm especificação definida
faltantes = set(series_macro) - set(especificacao.keys())
if faltantes:
    print(f"⚠ Sem especificação: {faltantes}")
else:
    print("✓ Todas as séries têm especificação definida")

✓ Todas as séries têm especificação definida


In [3]:
def testar_estacionariedade(serie, nome, regression):
    """
    Aplica ADF e KPSS em uma série e retorna um dicionário consolidado.
    
    Parameters
    ----------
    serie : pd.Series
        Série temporal já limpa (sem NaN)
    nome : str
        Identificador da série
    regression : str
        Especificação determinística: 'c' (constante) ou 'ct' (constante + tendência)
    
    Returns
    -------
    dict com resultados dos dois testes e classificação do caso
    """
    serie_limpa = serie.dropna()
    
    # ADF
    adf_stat, adf_pvalue, adf_lags, adf_nobs, _, _ = adfuller(
        serie_limpa, regression=regression, autolag='AIC'
    )
    
    # KPSS — usa 'c' ou 'ct' também, parâmetro chamado 'regression' aqui também
    kpss_stat, kpss_pvalue, kpss_lags, _ = kpss(
        serie_limpa, regression=regression, nlags='auto'
    )
    
    # Classificação dos casos (a 5% de significância)
    adf_rejeita = adf_pvalue < 0.05  # rejeita raiz unitária → série estacionária
    kpss_rejeita = kpss_pvalue < 0.05  # rejeita estacionariedade → série não-estacionária
    
    if adf_rejeita and not kpss_rejeita:
        caso = 'A — Estacionária'
    elif not adf_rejeita and kpss_rejeita:
        caso = 'B — Não-estacionária'
    elif adf_rejeita and kpss_rejeita:
        caso = 'C — Conflito (testar quebra)'
    else:
        caso = 'D — Inconclusivo'
    
    return {
        'serie': nome,
        'spec': regression,
        'adf_stat': adf_stat,
        'adf_pvalue': adf_pvalue,
        'adf_lags': adf_lags,
        'kpss_stat': kpss_stat,
        'kpss_pvalue': kpss_pvalue,
        'kpss_lags': kpss_lags,
        'caso': caso,
    }

In [4]:
print("=" * 80)
print("TESTES DE ESTACIONARIEDADE — SÉRIES EM NÍVEL")
print("=" * 80)

resultados_nivel = []
for col in series_macro:
    spec = especificacao[col]
    res = testar_estacionariedade(df[col], col, spec)
    resultados_nivel.append(res)
    print(f"\n{col:25s} (spec: {spec})")
    print(f"  ADF:  stat={res['adf_stat']:>8.3f}  p-valor={res['adf_pvalue']:.4f}  lags={res['adf_lags']}")
    print(f"  KPSS: stat={res['kpss_stat']:>8.3f}  p-valor={res['kpss_pvalue']:.4f}  lags={res['kpss_lags']}")
    print(f"  → {res['caso']}")

TESTES DE ESTACIONARIEDADE — SÉRIES EM NÍVEL

ln_ibcbr                  (spec: ct)
  ADF:  stat=  -2.197  p-valor=0.4915  lags=2
  KPSS: stat=   0.448  p-valor=0.0100  lags=10
  → B — Não-estacionária

ln_cambio                 (spec: ct)
  ADF:  stat=  -3.060  p-valor=0.1160  lags=1
  KPSS: stat=   0.460  p-valor=0.0100  lags=10
  → B — Não-estacionária

ln_commodities            (spec: ct)
  ADF:  stat=  -2.757  p-valor=0.2133  lags=1
  KPSS: stat=   0.492  p-valor=0.0100  lags=10
  → B — Não-estacionária

ln_m1                     (spec: ct)
  ADF:  stat=  -2.373  p-valor=0.3939  lags=10
  KPSS: stat=   0.301  p-valor=0.0100  lags=10
  → B — Não-estacionária

ln_prod_industrial        (spec: c)
  ADF:  stat=  -2.659  p-valor=0.0814  lags=4
  KPSS: stat=   0.637  p-valor=0.0193  lags=10
  → B — Não-estacionária

ln_credito_total          (spec: ct)
  ADF:  stat=  -3.256  p-valor=0.0738  lags=14
  KPSS: stat=   0.572  p-valor=0.0100  lags=10
  → B — Não-estacionária

ipca             

C:\Users\lucas\AppData\Local\Temp\ipykernel_18700\2291843935.py:26: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is smaller than the p-value returned.

  kpss_stat, kpss_pvalue, kpss_lags, _ = kpss(
C:\Users\lucas\AppData\Local\Temp\ipykernel_18700\2291843935.py:26: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is smaller than the p-value returned.

  kpss_stat, kpss_pvalue, kpss_lags, _ = kpss(
C:\Users\lucas\AppData\Local\Temp\ipykernel_18700\2291843935.py:26: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is smaller than the p-value returned.

  kpss_stat, kpss_pvalue, kpss_lags, _ = kpss(
C:\Users\lucas\AppData\Local\Temp\ipykernel_18700\2291843935.py:26: InterpolationWarning: The test statistic is outside of the range of p-values available

In [5]:
# Converte lista de dicionários em DataFrame para visualização tabular
df_resultados_nivel = pd.DataFrame(resultados_nivel)

# Reorganiza colunas e arredonda
df_resultados_nivel = df_resultados_nivel[[
    'serie', 'spec', 'adf_stat', 'adf_pvalue', 'kpss_stat', 'kpss_pvalue', 'caso'
]].round({'adf_stat': 3, 'adf_pvalue': 4, 'kpss_stat': 3, 'kpss_pvalue': 4})

print("\nResumo dos testes em nível:\n")
print(df_resultados_nivel.to_string(index=False))


Resumo dos testes em nível:

             serie spec  adf_stat  adf_pvalue  kpss_stat  kpss_pvalue                 caso
          ln_ibcbr   ct    -2.197      0.4915      0.448       0.0100 B — Não-estacionária
         ln_cambio   ct    -3.060      0.1160      0.460       0.0100 B — Não-estacionária
    ln_commodities   ct    -2.757      0.2133      0.492       0.0100 B — Não-estacionária
             ln_m1   ct    -2.373      0.3939      0.301       0.0100 B — Não-estacionária
ln_prod_industrial    c    -2.659      0.0814      0.637       0.0193 B — Não-estacionária
  ln_credito_total   ct    -3.256      0.0738      0.572       0.0100 B — Não-estacionária
              ipca    c    -9.529      0.0000      0.116       0.1000     A — Estacionária
             selic    c    -1.781      0.3901      0.939       0.0100 B — Não-estacionária
      exp_ipca_12m    c    -2.481      0.1202      0.520       0.0371 B — Não-estacionária
